# Pick-and-Place Submission Tester

This notebook evaluates your `student_policy.py` on environment files and reports a **success rate**.

## What YOU (students) should edit
1. `student_policy.py` — your policy architecture, `build_policy`, and `run_single_episode`.
2. **Cell 3** config values — file paths, `MAX_ENVS`, etc.

## What you must NOT edit
- All other cells in this notebook (scene construction, evaluation loop, success counting).

## Success Criterion
An episode is **successful** if the final block position is within `SUCCESS_DISTANCE` (5 cm) of `target_xpos`.

## Grading
The grader runs this notebook with `ENV_DIR = './secret_env'` and `MAX_ENVS = None`.  
Your final score is the **success rate across all secret environments**.  
Make sure your policy generalizes — do not overfit to `open_env`.

## Notebook Flow
1. **Cell 2** — imports (no edits needed)
2. **Cell 3** — ⬅ **edit your paths here**
3. **Cells 4–7** — evaluator helpers (do not edit)
4. **Cell 8** — ▶ run this to evaluate and see your score


In [1]:
import os
# os.environ['MUJOCO_GL'] = 'egl'
# os.environ['PYOPENGL_PLATFORM'] = 'egl'

import importlib.util
import re
from pathlib import Path

import mujoco
import numpy as np
import torch
from robot_descriptions import panda_mj_description
from tqdm.auto import tqdm

from helper import reset_to_keyframe

In [2]:
# ====================== STUDENT CONFIG — EDIT THIS CELL ONLY ======================
#
# POLICY_FILE   : Path to your submission Python file (the one with POLICY and
#                 run_single_episode). Change this to './student_policy.py' when
#                 submitting your own work. The default points to the provided
#                 baseline so you can verify the tester runs correctly first.
POLICY_FILE = './student_policy.py'  # <-- change to your file

# CHECKPOINT_FILE: Path to your trained model weights (.pth). This is the file
#                  saved by torch.save(model.state_dict(), 'your_model.pth').
#                  Set to None if you are NOT loading a checkpoint.
CHECKPOINT_FILE = './rnn_transition_random800_dagger80_best.pth'  # <-- change to your checkpoint, or None

# STATS_FILE     : Path to your normalization stats (.npz) produced by
#                  baseline_trainer.py. Set to None if you do NOT use normalization.
STATS_FILE = './rnn_transition_random800_dagger80_stats.npz'      # <-- change to your stats file, or None

# ENV_DIR        : Directory containing evaluation environment .npz files.
#                  Use './open_env' for local testing.
#                  The grader will run with a private secret_env directory.
ENV_DIR = './open_env'

# MAX_ENVS       : Number of environments to evaluate.
#                  Set to a small number (e.g. 5) for a quick smoke test,
#                  then set to None to evaluate ALL environments for final grading.
MAX_ENVS = None  # <-- final grading will be made by setting this value as None

# ── Do not change these unless you have a good reason ──
MAX_STEPS           = 5000    # max simulation steps per episode
RENDER_ONE_AT_END   = True    # render a sample episode after bulk eval
RENDER_SELECTION_SEED = 42
RENDER_FPS          = 60
# ===================================================================================


## Before Running
1. Edit the paths/options in **Cell 3**.
2. Your policy file must define:
   - `POLICY(nn.Module)` — with a working `forward(x: dict) -> Tensor[B,8]`
   - `build_policy(checkpoint_path, device, ...)` — builds and loads your model
   - `run_single_episode(policy, model, data, ids, env_data, stats, device, ...)` — runs one episode
3. Run **all cells top to bottom**.

## What Your `run_single_episode` Receives
| Argument | Type | Description |
|---|---|---|
| `policy` | `nn.Module` | Your trained model (eval mode, on `device`) |
| `model` | `MjModel` | MuJoCo model for this environment |
| `data` | `MjData` | Mutable sim state — **write to `data.ctrl` every step** |
| `ids` | `dict` | `block_body_id`, `hand_body_id`, `task_cam_id` |
| `env_data` | `dict` | Fields from the env `.npz`: `target_xpos`, `block_size`, `block_init_xpos`, `block_init_quat` |
| `stats` | `npz` or `None` | Normalization stats from your training pipeline |
| `device` | `torch.device` | cuda or cpu |

## Common Errors
| Error | Likely Cause | Fix |
|---|---|---|
| `FileNotFoundError` | Wrong path in Cell 3 | Double-check all file paths |
| Shape mismatch in policy input | Observation layout differs from training | Ensure observation keys/dims match `baseline_trainer.py` |
| `KeyError: in_<key>_mean` | Feature in `model_input` not in stats | Use the same feature set in training and inference |
| Good train loss, poor rollout | Normalization mismatch or wrong feature order | Check `STATS_FILE` and that keys are sorted consistently |
| Gripper never closes | Gripper channel decoding issue | Verify `pred[7]` range and threshold (~128) |


In [3]:
def load_student_module(policy_file: str):
    policy_path = Path(policy_file).resolve()
    if not policy_path.exists():
        raise FileNotFoundError(f'Policy file not found: {policy_path}')

    spec = importlib.util.spec_from_file_location('student_submission_policy', str(policy_path))
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


def build_policy_from_module(module, checkpoint_file: str, device: torch.device):
    checkpoint_path = Path(checkpoint_file).resolve() if checkpoint_file else None
    if checkpoint_path is not None and (not checkpoint_path.exists()):
        raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')

    if hasattr(module, 'build_policy'):
        ckpt_arg = str(checkpoint_path) if checkpoint_path is not None else None
        policy = module.build_policy(ckpt_arg, device)
    elif hasattr(module, 'POLICY'):
        policy = module.POLICY(input_dim=53, output_dim=8).to(device)
        if checkpoint_path is not None:
            state_dict = torch.load(str(checkpoint_path), map_location=device)
            policy.load_state_dict(state_dict)
        policy.eval()
    else:
        raise AttributeError('Submission must define POLICY class or build_policy function.')

    return policy


def validate_student_api(module):
    if not hasattr(module, 'run_single_episode'):
        raise AttributeError(
            "Submission must define run_single_episode(policy, model, data, ids, env_data, stats, device, max_steps, render, render_fps, success_distance)."
        )

In [4]:
target_radius = 0.05

def build_scene_xml(block_size, block_pos, block_quat, target_pos):
    with open(panda_mj_description.MJCF_PATH, 'r') as f:
        base_xml = f.read()

    block_x, block_y, block_z = block_pos
    table_top_z = block_z - block_size
    table_x, table_y = 0.58, 0.10
    table_size_x, table_size_y = 0.25, 0.35
    # target_radius = 0.05
    target_x, target_y = target_pos[0], target_pos[1]
    target_z = table_top_z + 0.001

    extras = f"""
    <geom name='floor' type='plane' size='2 2 0.05' rgba='0.85 0.85 0.9 1'
            friction='1.0 0.05 0.001'/>

    <geom name='table' type='box' pos='{table_x} {table_y} 0.0' size='{table_size_x} {table_size_y} {table_top_z}'
            rgba='0.75 0.65 0.5 1' friction='1.0 0.05 0.001'/>

    <body name='block' pos='{block_x} {block_y} {block_z}' quat='{block_quat[0]} {block_quat[1]} {block_quat[2]} {block_quat[3]}'>
        <freejoint name='block_free'/>
        <geom name='block_geom' type='box' size='{block_size} {block_size} {block_size}'
            rgba='0.9 0.2 0.2 1' mass='0.05'
            friction='1.0 0.05 0.001'/>
    </body>

    <body name='target_marker' mocap='true' pos='{target_x} {target_y} {target_z}'>
        <geom type='cylinder' size='{target_radius} 0.001' rgba='0.2 0.9 0.2 0.5'
            contype='0' conaffinity='0'/>
    </body>

    <camera name='task_view' pos='1.6 -0.6 1.0' xyaxes='0.5 0.87 0  -0.30 0.17 0.94'/>
    """

    patched_xml = base_xml.replace('</worldbody>', extras + '\n  </worldbody>')
    patched_xml = re.sub(
        r'(<key\s+name="home"\s+qpos=")([^"]+)(")',
        r'\1\2 ' + f'{block_x} {block_y} {block_z} ' + f'{block_quat[0]} {block_quat[1]} {block_quat[2]} {block_quat[3]}' + r'\3',
        patched_xml,
    )

    return patched_xml


def build_model_data_for_env(env_npz_path: str):
    raw = np.load(env_npz_path)
    env_data = {k: raw[k] for k in raw.files}
    block_size = float(env_data['block_size'])
    block_pos = env_data['block_init_xpos']
    block_quat = env_data['block_init_quat']
    target_pos = env_data['target_xpos']

    patched_xml = build_scene_xml(block_size, block_pos, block_quat, target_pos)
    tmp_path = Path(panda_mj_description.PACKAGE_PATH) / '_submission_eval_scene.xml'
    with open(tmp_path, 'w') as f:
        f.write(patched_xml)

    model = mujoco.MjModel.from_xml_path(str(tmp_path))
    data = mujoco.MjData(model)
    reset_to_keyframe(model, data, 'home')

    ids = {
        'block_body_id': mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, 'block'),
        'hand_body_id': mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, 'hand'),
        'task_cam_id': mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_CAMERA, 'task_view'),
    }

    return model, data, ids, env_data

In [5]:
def evaluate_single_env(student_module, policy, stats, env_npz_path, device, render=False):
    model, data, ids, env_data = build_model_data_for_env(env_npz_path)

    success_distance = target_radius + env_data['block_size']   # block must be within this distance of target to count as success
    # Run student-defined rollout. Success is computed by evaluator code below.
    student_out = student_module.run_single_episode(
        policy=policy,
        model=model,
        data=data,
        ids=ids,
        env_data=env_data,
        stats=stats,
        device=device,
        max_steps=MAX_STEPS,
        render=render,
        render_fps=RENDER_FPS,
        success_distance=success_distance,
    )

    final_block_pos = data.xpos[ids['block_body_id']].copy()
    final_dist = np.linalg.norm(final_block_pos - env_data['target_xpos'])
    success = final_dist < success_distance

    return {
        'success': bool(success),
        'final_distance': float(final_dist),
        'final_block_pos': final_block_pos,
        'target_pos': env_data['target_xpos'],
        'student_output': student_out
    }

## How to Read the Output
- `Success: x/y` — number of successful episodes out of total evaluated.
- `Success rate: XX.XX%` — **this is your score**.
- `Sample failures` — lists environments where the block ended farthest from the target (useful for debugging).
- The rendered video at the end is **one sample only** — always use the aggregate success rate as your primary metric.

## Final Submission Checklist
- [ ] `POLICY_FILE` points to your final `student_policy.py`.
- [ ] `CHECKPOINT_FILE` points to the exact `.pth` used for these results.
- [ ] `STATS_FILE` matches the normalization used during training (or `None` if unused).
- [ ] `MAX_ENVS = None` — evaluated on ALL environments, not just a subset.
- [ ] `run_single_episode` completes without errors on the full environment set.
- [ ] You are submitting `student_policy.py` (and `.pth` + `.npz`), **not** this notebook.


In [ ]:
# 1) Load student submission and optional checkpoint/stats.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

student_module = load_student_module(POLICY_FILE)
validate_student_api(student_module)
policy = build_policy_from_module(student_module, CHECKPOINT_FILE, device)
stats = np.load(STATS_FILE) if STATS_FILE else None

# 2) Collect evaluation environments.
env_paths = sorted(str(p) for p in Path(ENV_DIR).glob('*.npz'))
if len(env_paths) == 0:
    raise RuntimeError(f'No .npz environment files found under {ENV_DIR}')

if MAX_ENVS is not None:
    env_paths = env_paths[:MAX_ENVS]

print(f'Evaluating {len(env_paths)} environments from {ENV_DIR}')

# 3) Run one rollout per environment (no rendering during bulk evaluation).
results = []
for i, env_path in enumerate(tqdm(env_paths, desc='Evaluating')):
    outcome = evaluate_single_env(student_module, policy, stats, env_path, device, render=False)
    outcome['env_path'] = env_path
    print(f"Env {i+1}/{len(env_paths)}: {Path(env_path).name} -> success={outcome['success']}, final_dist={outcome['final_distance']:.4f} m")
    results.append(outcome)

# 4) Summarize success rate.
success_count = sum(int(r['success']) for r in results)
total_count = len(results)
success_rate = success_count / total_count

print('----------------------------------------')
print(f'Success: {success_count}/{total_count}')
print(f'Success rate: {success_rate * 100:.2f}%')
print('----------------------------------------')

# 5) Show sample failures for debugging.
failures = [r for r in results if not r['success']]
if len(failures) > 0:
    print('Sample failures (up to 5):')
    for r in failures[:5]:
        print(f"- {Path(r['env_path']).name}: final_dist={r['final_distance']:.4f} m")
else:
    print('All environments succeeded.')

# 6) Render exactly one environment at the end.
if RENDER_ONE_AT_END:
    np.random.seed(RENDER_SELECTION_SEED)
    successful_env_paths = [r['env_path'] for r in results if r['success']]

    if len(successful_env_paths) > 0:
        picked_env_path = str(np.random.choice(successful_env_paths))
        print(f'Rendering one successful environment: {Path(picked_env_path).name}')
    else:
        picked_env_path = str(np.random.choice(env_paths))
        print(f'No successful environment found. Rendering a random environment: {Path(picked_env_path).name}')

    render_outcome = evaluate_single_env(student_module, policy, stats, picked_env_path, device, render=True)
    print(
        f"Rendered env result -> success={render_outcome['success']}, "
        f"final_dist={render_outcome['final_distance']:.4f} m"
    )

device: cuda
Evaluating 250 environments from ./open_env


Evaluating:   0%|          | 0/250 [00:00<?, ?it/s]

Env 1/250: 0.npz -> success=True, final_dist=0.0350 m
Env 2/250: 1.npz -> success=False, final_dist=0.1493 m
Env 3/250: 10.npz -> success=True, final_dist=0.0388 m
Env 4/250: 100.npz -> success=True, final_dist=0.0287 m
Env 5/250: 101.npz -> success=True, final_dist=0.0449 m
Env 6/250: 102.npz -> success=True, final_dist=0.0267 m
Env 7/250: 103.npz -> success=True, final_dist=0.0237 m
Env 8/250: 104.npz -> success=True, final_dist=0.0505 m
Env 9/250: 105.npz -> success=True, final_dist=0.0214 m
Env 10/250: 106.npz -> success=False, final_dist=0.1226 m
Env 11/250: 107.npz -> success=False, final_dist=0.1087 m
Env 12/250: 108.npz -> success=True, final_dist=0.0317 m
